# DuckDuckGo
- 실시간 웹 검색 구현
- 트래픽 급증 속 'No-AI' 검색 엔진에 더  쉽게 접근할 수 있게 함
- https://duckduckgo.com/

In [2]:
!pip install -U langchain-openai langchain-community
!pip install duckduckgo-search ddgs # DuckDuckGo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.8
    Uninstalling langchain-core-1.4.8:
      Successfully uninstalled langchain-core-1.4.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-

In [5]:
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.prompts import ChatPromptTemplate
import time
import os

from dotenv import load_dotenv

load_dotenv()

class MyRealtimeWebRag:
  def __init__(self):
    # 검색 도구 생성
    self.search = DuckDuckGoSearchResults()
    self.llm = ChatOpenAI(
        model = 'gpt-4o-mini',
        temperature=0.0, # 일관된 답변을 생성(정형적)
        api_key=os.getenv("OPENAI_API_KEY")
    )
    # prompt
    message = """
      web에서 검색한 최신정보를 바탕으로 답변하세요.

      [검색 결과]
      {search_results}

      [질문]
      {question}

      [중요]
      검색 결과에 있는 답변만 하세요. 추측하지 마세요.
      모르면 모른다고 하세요.

      [답변]
    """

    self.qa_prompt = ChatPromptTemplate.from_messages(
        [
            ("human", message), # 'role' : 'user' 와 유사한 의미
        ]
    )

  def answerFunc(self, question):
    print(f'검색 중 : {question}')
    # 프롬프트 내용 보강용 데이터 얻기
    search_results = self.search.run(question) # DuckDuckGo로 질문 관련 웹 검색 실행
    # 웹검색 텀 주기
    time.sleep(1)
    qa_chain = self.qa_prompt | self.llm # prompt와 llm을 체인으로 연결

    response = qa_chain.invoke( # 1) invoke : LLM에게 prompt에 대한 답변 요청 (총 invoke가 3번 나옴)
        {
        'search_results': search_results, # 검색 결과를 prompt의 {search_results}에 전달
        'question': question, # 사용자 질문을 prompt의 {question}에 전달
      }
    )
    return response

if __name__=="__main__":
  web_rag = MyRealtimeWebRag()
  questions = [
      "최신 AI 기술은 뭐니?",
      "한국에서 가장 인기 있는 빵은?",
      "한국의 전망있는 의료AI 관련 제품을 소개해"
  ]

  for q in questions:
    print(f"질문:'{q}'")
    answer = web_rag.answerFunc(q)
    print(f'[답변] \n{answer.content}\n')

질문:'최신 AI 기술은 뭐니?'
검색 중 : 최신 AI 기술은 뭐니?
[답변] 
최신 AI 기술 중 하나는 생성형 AI로, 이는 딥러닝 발전과 함께 최근 몇 년 사이 폭발적으로 성장했습니다. 특히 2022년 말 공개된 OpenAI의 ChatGPT가 결정적인 전환점이 되었습니다. 생성형 AI는 글쓰기와 관련된 작업에서 높은 활용도를 보이며, 질문의 의도를 세밀하게 설정할수록 더 정교한 결과물을 얻을 수 있습니다. 이를 프롬프트 엔지니어링이라고 부르며, 인공지능과의 소통 방식을 익히는 것이 중요해졌습니다. 최근에는 실시간으로 인터넷 정보를 검색하여 최신 데이터를 반영한 답변을 제공하는 기능도 포함되고 있습니다.

질문:'한국에서 가장 인기 있는 빵은?'
검색 중 : 한국에서 가장 인기 있는 빵은?
[답변] 
현재 한국에서 가장 인기 있는 빵 종류에 대한 정보는 다음과 같습니다:

1. **크림빵**: 달콤하고 부드러운 크림으로 가득 찬 속이 있는 빵으로, 한국인에게 매우 인기 있는 간식입니다.
2. **단팥빵**: 부드러운 빵 속에 달콤한 팥앙금이 들어간 전통 인기 빵입니다. 일본의 안팡(Anpan) 영향을 받아 1950년대 한국화된 버전입니다.

더 많은 인기 빵 종류에 대한 정보는 [여기](https://blog.naver.com/number1220/224037509983)에서 확인할 수 있습니다.

질문:'한국의 전망있는 의료AI 관련 제품을 소개해'
검색 중 : 한국의 전망있는 의료AI 관련 제품을 소개해
[답변] 
한국의 전망 있는 의료 AI 관련 제품으로는 다음과 같은 것들이 있습니다:

1. **제이엘케이 의료 AI 플랫폼**: 제이엘케이는 AI 기반 영상진단 솔루션을 제공하는 기업으로, AI 원천 알고리즘 기술을 바탕으로 의료 분야 최초로 코스닥에 상장되었습니다. 이 플랫폼은 인공지능을 활용하여 의료 영상 및 임상 데이터를 분석하고 질병을 조기에 진단하는 데 도움을 줍니다.

2. **AI 웰니스 홈**: 세라젬은 CES에서 헬스케어 중심의 'AI